In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import pandas as pd
import numpy as np
import os
from google.colab import files

In [19]:
# 1. 데이터 파일 로드

import pandas as pd
import numpy as np

file_path = "/content/drive/MyDrive/♦︎OBS_계절관측_벚나무.xlsx"

# 벚나무 원자료는 앞 2행이 다중 헤더이므로 header=None으로 불러옴
raw = pd.read_excel(file_path, header=None)

# 원본 확인
print("원본 데이터 크기:", raw.shape)
display(raw.head())

원본 데이터 크기: (229, 8)


,0,1,2,3,4,5,6,7
0,지점,년도,벚나무,벚나무,벚나무,벚나무,벚나무,벚나무
1,NaN,NaN,발아,발아(평비),개화,개화(평비),만발,만발(평비)
2,북강릉,2016,2016-03-19 00:00:00,-5일,2016-04-04 00:00:00,0일,2016-04-06 00:00:00,-1일
3,북강릉,2017,2017-03-19 00:00:00,-5일,2017-04-03 00:00:00,-1일,2017-04-04 00:00:00,-3일
4,북강릉,2018,2018-03-22 00:00:00,-2일,2018-03-31 00:00:00,-4일,2018-04-02 00:00:00,-5일


In [20]:
# 2. 컬럼 정리 및 실제 관측값 추출

# 앞 2행 제거 후 실제 관측값만 사용
df = raw.iloc[2:].reset_index(drop=True)

# 앞의 8개 컬럼만 사용
df = df.iloc[:, :8]

# 컬럼명 표준화
df.columns = [
    "Location",
    "Year",
    "Bud_Date",
    "Bud_Gap",
    "Bloom_Dt",
    "Bloom_Gap",
    "Full_Dt",
    "Full_Gap"
]

# 빈 행 제거
df = df.dropna(how="all").reset_index(drop=True)

# Location명 공백 제거
df["Location"] = df["Location"].astype(str).str.strip()

# 확인
print("정리 후 데이터 크기:", df.shape)
display(df.head())

정리 후 데이터 크기: (227, 8)


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap
0,북강릉,2016,2016-03-19 00:00:00,-5일,2016-04-04 00:00:00,0일,2016-04-06 00:00:00,-1일
1,북강릉,2017,2017-03-19 00:00:00,-5일,2017-04-03 00:00:00,-1일,2017-04-04 00:00:00,-3일
2,북강릉,2018,2018-03-22 00:00:00,-2일,2018-03-31 00:00:00,-4일,2018-04-02 00:00:00,-5일
3,북강릉,2019,2019-03-20 00:00:00,-4일,2019-03-31 00:00:00,-4일,2019-04-04 00:00:00,-3일
4,북강릉,2020,2020-03-21 00:00:00,-3일,2020-03-26 00:00:00,-9일,2020-03-30 00:00:00,-8일


In [21]:
# 3. 자료형 변환

# Year 숫자형 변환
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

# 날짜형 변환
for col in ["Bud_Date", "Bloom_Dt", "Full_Dt"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# 평년비 숫자형 변환
# 예: "-3일", "+2일", "0일" -> -3, 2, 0
for col in ["Bud_Gap", "Bloom_Gap", "Full_Gap"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("일", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.strip()
        .replace(["nan", "", "-", "None"], np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("자료형 변환 완료")
print(df.dtypes)
display(df.head())

자료형 변환 완료
Location             object
Year                  int64
Bud_Date     datetime64[ns]
Bud_Gap             float64
Bloom_Dt     datetime64[ns]
Bloom_Gap           float64
Full_Dt      datetime64[ns]
Full_Gap            float64
dtype: object


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap
0,북강릉,2016,2016-03-19,-5.0,2016-04-04,0.0,2016-04-06,-1.0
1,북강릉,2017,2017-03-19,-5.0,2017-04-03,-1.0,2017-04-04,-3.0
2,북강릉,2018,2018-03-22,-2.0,2018-03-31,-4.0,2018-04-02,-5.0
3,북강릉,2019,2019-03-20,-4.0,2019-03-31,-4.0,2019-04-04,-3.0
4,북강릉,2020,2020-03-21,-3.0,2020-03-26,-9.0,2020-03-30,-8.0


In [22]:
# 4. 결측치 확인

missing = df.isna().sum().reset_index()
missing.columns = ["컬럼", "결측치_개수"]
missing["결측치_비율(%)"] = (
    missing["결측치_개수"] / len(df) * 100
).round(2)

display(missing)

print("전체 결측치 총합:", df.isna().sum().sum())

,컬럼,결측치_개수,결측치_비율(%)
0,Location,0,0.00
1,Year,0,0.00
2,Bud_Date,0,0.00
3,Bud_Gap,9,3.96
4,Bloom_Dt,0,0.00
5,Bloom_Gap,9,3.96
6,Full_Dt,0,0.00
7,Full_Gap,68,29.96


전체 결측치 총합: 86


In [23]:
# 5. 날짜 분석용 / 평년비 분석용 데이터 분리

date_required_cols = [
    "Location",
    "Year",
    "Bud_Date",
    "Bloom_Dt",
    "Full_Dt"
]

anomaly_cols = [
    "Bud_Gap",
    "Bloom_Gap",
    "Full_Gap"
]

# 날짜 핵심 컬럼 결측 확인
date_missing_rows = df[df[date_required_cols].isna().any(axis=1)]

print("날짜 핵심 컬럼 결측 행 수:", len(date_missing_rows))
display(date_missing_rows)

# 날짜 분석용 데이터
# 날짜 정보가 있으면 유지
df_date = df.dropna(subset=date_required_cols).copy()
df_date["Year"] = df_date["Year"].astype(int)

# 평비 결측 여부 표시
for col in anomaly_cols:
    df_date[col + "_결측여부"] = df_date[col].isna()

# 평년비 분석용 데이터
# 평비 결측값은 임의 대체하지 않고 제외
df_anomaly = df_date.dropna(subset=anomaly_cols).copy()

print("날짜 분석용 데이터 크기:", df_date.shape)
print("평년비 분석용 데이터 크기:", df_anomaly.shape)

print("평비 결측으로 평년비 분석에서 제외될 행")
display(df_date[df_date[anomaly_cols].isna().any(axis=1)])

날짜 핵심 컬럼 결측 행 수: 0


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap


날짜 분석용 데이터 크기: (227, 11)
평년비 분석용 데이터 크기: (159, 11)
평비 결측으로 평년비 분석에서 제외될 행


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap,Bud_Gap_결측여부,Bloom_Gap_결측여부,Full_Gap_결측여부
19,춘천,2016,2016-03-27,NaN,2016-04-05,NaN,2016-04-07,NaN,True,True,True
100,부산,2016,2016-03-09,-4.0,2016-03-28,0.0,2016-04-05,NaN,False,False,True
101,부산,2017,2017-03-16,3.0,2017-03-28,0.0,2017-04-01,NaN,False,False,True
102,부산,2018,2018-03-10,-3.0,2018-03-27,-1.0,2018-03-30,NaN,False,False,True
103,부산,2019,2019-03-13,0.0,2019-03-22,-6.0,2019-03-28,NaN,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...
212,홍성,2021,2021-03-23,NaN,2021-03-31,NaN,2021-04-04,NaN,True,True,True
213,홍성,2022,2022-04-01,NaN,2022-04-08,NaN,2022-04-11,NaN,True,True,True
214,홍성,2023,2023-03-24,NaN,2023-03-30,NaN,2023-04-01,NaN,True,True,True
215,홍성,2024,2024-03-29,NaN,2024-04-07,NaN,2024-04-11,NaN,True,True,True


In [24]:
# 6. DOY 및 유지기간 생성

for data in [df_date, df_anomaly]:
    data["Bud_DOY"] = data["Bud_Date"].dt.dayofyear
    data["Bloom_DOY"] = data["Bloom_Dt"].dt.dayofyear
    data["Full_DOY"] = data["Full_Dt"].dt.dayofyear

    # 꽃 유지기간 (만발일 - 개화일)
    data["Flowering_Duration"] = data["Full_DOY"] - data["Bloom_DOY"]

display(df_date.head())
display(df_anomaly.head())


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap,Bud_Gap_결측여부,Bloom_Gap_결측여부,Full_Gap_결측여부,Bud_DOY,Bloom_DOY,Full_DOY,Flowering_Duration
0,북강릉,2016,2016-03-19,-5.0,2016-04-04,0.0,2016-04-06,-1.0,False,False,False,79,95,97,2
1,북강릉,2017,2017-03-19,-5.0,2017-04-03,-1.0,2017-04-04,-3.0,False,False,False,78,93,94,1
2,북강릉,2018,2018-03-22,-2.0,2018-03-31,-4.0,2018-04-02,-5.0,False,False,False,81,90,92,2
3,북강릉,2019,2019-03-20,-4.0,2019-03-31,-4.0,2019-04-04,-3.0,False,False,False,79,90,94,4
4,북강릉,2020,2020-03-21,-3.0,2020-03-26,-9.0,2020-03-30,-8.0,False,False,False,81,86,90,4


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap,Bud_Gap_결측여부,Bloom_Gap_결측여부,Full_Gap_결측여부,Bud_DOY,Bloom_DOY,Full_DOY,Flowering_Duration
0,북강릉,2016,2016-03-19,-5.0,2016-04-04,0.0,2016-04-06,-1.0,False,False,False,79,95,97,2
1,북강릉,2017,2017-03-19,-5.0,2017-04-03,-1.0,2017-04-04,-3.0,False,False,False,78,93,94,1
2,북강릉,2018,2018-03-22,-2.0,2018-03-31,-4.0,2018-04-02,-5.0,False,False,False,81,90,92,2
3,북강릉,2019,2019-03-20,-4.0,2019-03-31,-4.0,2019-04-04,-3.0,False,False,False,79,90,94,4
4,북강릉,2020,2020-03-21,-3.0,2020-03-26,-9.0,2020-03-30,-8.0,False,False,False,81,86,90,4


In [25]:
# 7. 날짜 순서 오류 확인 및 제거
# 기준: Bud_Date ≤ Bloom_Dt ≤ Full_Dt

logic_error_date = df_date[
    (df_date["Bloom_Dt"] < df_date["Bud_Date"]) |
    (df_date["Full_Dt"] < df_date["Bloom_Dt"]) |
    (df_date["Full_Dt"] < df_date["Bud_Date"])
]

logic_error_anomaly = df_anomaly[
    (df_anomaly["Bloom_Dt"] < df_anomaly["Bud_Date"]) |
    (df_anomaly["Full_Dt"] < df_anomaly["Bloom_Dt"]) |
    (df_anomaly["Full_Dt"] < df_anomaly["Bud_Date"])
]

print("날짜 분석용 날짜 순서 오류:", len(logic_error_date))
display(logic_error_date)

print("평년비 분석용 날짜 순서 오류:", len(logic_error_anomaly))
display(logic_error_anomaly)

# 오류 제거
df_date = df_date[
    ~(
        (df_date["Bloom_Dt"] < df_date["Bud_Date"]) |
        (df_date["Full_Dt"] < df_date["Bloom_Dt"]) |
        (df_date["Full_Dt"] < df_date["Bud_Date"])
    )
].copy()

df_anomaly = df_anomaly[
    ~(
        (df_anomaly["Bloom_Dt"] < df_anomaly["Bud_Date"]) |
        (df_anomaly["Full_Dt"] < df_anomaly["Bloom_Dt"]) |
        (df_anomaly["Full_Dt"] < df_anomaly["Bud_Date"])
    )
].copy()

날짜 분석용 날짜 순서 오류: 0


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap,Bud_Gap_결측여부,Bloom_Gap_결측여부,Full_Gap_결측여부,Bud_DOY,Bloom_DOY,Full_DOY,Flowering_Duration


평년비 분석용 날짜 순서 오류: 0


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap,Bud_Gap_결측여부,Bloom_Gap_결측여부,Full_Gap_결측여부,Bud_DOY,Bloom_DOY,Full_DOY,Flowering_Duration


In [26]:
# 8. DOY 이상치 확인 및 제거

doy_outlier_date = df_date[
    (df_date["Bud_DOY"] < 40) | (df_date["Bud_DOY"] > 120) |
    (df_date["Bloom_DOY"] < 50) | (df_date["Bloom_DOY"] > 130) |
    (df_date["Full_DOY"] < 55) | (df_date["Full_DOY"] > 140)
]

doy_outlier_anomaly = df_anomaly[
    (df_anomaly["Bud_DOY"] < 40) | (df_anomaly["Bud_DOY"] > 120) |
    (df_anomaly["Bloom_DOY"] < 50) | (df_anomaly["Bloom_DOY"] > 130) |
    (df_anomaly["Full_DOY"] < 55) | (df_anomaly["Full_DOY"] > 140)
]

print("날짜 분석용 DOY 이상치:", len(doy_outlier_date))
display(doy_outlier_date)

print("평년비 분석용 DOY 이상치:", len(doy_outlier_anomaly))
display(doy_outlier_anomaly)

# 이상치 제거
df_date = df_date[
    (df_date["Bud_DOY"].between(40, 120)) &
    (df_date["Bloom_DOY"].between(50, 130)) &
    (df_date["Full_DOY"].between(55, 140))
].copy()

df_anomaly = df_anomaly[
    (df_anomaly["Bud_DOY"].between(40, 120)) &
    (df_anomaly["Bloom_DOY"].between(50, 130)) &
    (df_anomaly["Full_DOY"].between(55, 140))
].copy()

날짜 분석용 DOY 이상치: 0


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap,Bud_Gap_결측여부,Bloom_Gap_결측여부,Full_Gap_결측여부,Bud_DOY,Bloom_DOY,Full_DOY,Flowering_Duration


평년비 분석용 DOY 이상치: 0


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap,Bud_Gap_결측여부,Bloom_Gap_결측여부,Full_Gap_결측여부,Bud_DOY,Bloom_DOY,Full_DOY,Flowering_Duration


In [27]:
# 9. 중복 확인
# 기준: Location + Year

dup_date = df_date[df_date.duplicated(subset=["Location", "Year"], keep=False)]
dup_anomaly = df_anomaly[df_anomaly.duplicated(subset=["Location", "Year"], keep=False)]

print("날짜 분석용 중복 행 수:", len(dup_date))
display(dup_date)

print("평년비 분석용 중복 행 수:", len(dup_anomaly))
display(dup_anomaly)

날짜 분석용 중복 행 수: 0


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap,Bud_Gap_결측여부,Bloom_Gap_결측여부,Full_Gap_결측여부,Bud_DOY,Bloom_DOY,Full_DOY,Flowering_Duration


평년비 분석용 중복 행 수: 0


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap,Bud_Gap_결측여부,Bloom_Gap_결측여부,Full_Gap_결측여부,Bud_DOY,Bloom_DOY,Full_DOY,Flowering_Duration


In [28]:
# 10. 연도별 정렬

df_date = df_date.sort_values(by=["Year", "Location"]).reset_index(drop=True)
df_anomaly = df_anomaly.sort_values(by=["Year", "Location"]).reset_index(drop=True)

display(df_date.head())
display(df_anomaly.head())

,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap,Bud_Gap_결측여부,Bloom_Gap_결측여부,Full_Gap_결측여부,Bud_DOY,Bloom_DOY,Full_DOY,Flowering_Duration
0,광주,2016,2016-03-13,0.0,2016-03-29,-2.0,2016-03-31,-4.0,False,False,False,73,89,91,2
1,대구,2016,2016-03-22,6.0,2016-03-25,-4.0,2016-03-31,-2.0,False,False,False,82,85,91,6
2,대전,2016,2016-03-19,-6.0,2016-03-31,-4.0,2016-04-02,-3.0,False,False,False,79,91,93,2
3,목포,2016,2016-03-14,-3.0,2016-03-31,-3.0,2016-04-02,-5.0,False,False,False,74,91,93,2
4,백령도,2016,2016-04-11,8.0,2016-04-17,-2.0,2016-04-20,NaN,False,False,True,102,108,111,3


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap,Bud_Gap_결측여부,Bloom_Gap_결측여부,Full_Gap_결측여부,Bud_DOY,Bloom_DOY,Full_DOY,Flowering_Duration
0,광주,2016,2016-03-13,0.0,2016-03-29,-2.0,2016-03-31,-4.0,False,False,False,73,89,91,2
1,대구,2016,2016-03-22,6.0,2016-03-25,-4.0,2016-03-31,-2.0,False,False,False,82,85,91,6
2,대전,2016,2016-03-19,-6.0,2016-03-31,-4.0,2016-04-02,-3.0,False,False,False,79,91,93,2
3,목포,2016,2016-03-14,-3.0,2016-03-31,-3.0,2016-04-02,-5.0,False,False,False,74,91,93,2
4,북강릉,2016,2016-03-19,-5.0,2016-04-04,0.0,2016-04-06,-1.0,False,False,False,79,95,97,2


In [29]:
# 11. 최종 점검

print("===== 날짜 분석용 최종 데이터 =====")
print("데이터 크기:", df_date.shape)
print("연도 범위:", df_date["Year"].min(), "~", df_date["Year"].max())
print("Location 수:", df_date["Location"].nunique())
print("전체 결측치:", df_date.isna().sum().sum())
display(df_date.isna().sum())
display(df_date.head())

print("===== 평년비 분석용 최종 데이터 =====")
print("데이터 크기:", df_anomaly.shape)
print("연도 범위:", df_anomaly["Year"].min(), "~", df_anomaly["Year"].max())
print("Location 수:", df_anomaly["Location"].nunique())
print("전체 결측치:", df_anomaly.isna().sum().sum())
display(df_anomaly.isna().sum())
display(df_anomaly.head())

===== 날짜 분석용 최종 데이터 =====
데이터 크기: (227, 15)
연도 범위: 2016 ~ 2025
Location 수: 25
전체 결측치: 86


,0
Location,0
Year,0
Bud_Date,0
Bud_Gap,9
Bloom_Dt,0
Bloom_Gap,9
Full_Dt,0
Full_Gap,68
Bud_Gap_결측여부,0
Bloom_Gap_결측여부,0


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap,Bud_Gap_결측여부,Bloom_Gap_결측여부,Full_Gap_결측여부,Bud_DOY,Bloom_DOY,Full_DOY,Flowering_Duration
0,광주,2016,2016-03-13,0.0,2016-03-29,-2.0,2016-03-31,-4.0,False,False,False,73,89,91,2
1,대구,2016,2016-03-22,6.0,2016-03-25,-4.0,2016-03-31,-2.0,False,False,False,82,85,91,6
2,대전,2016,2016-03-19,-6.0,2016-03-31,-4.0,2016-04-02,-3.0,False,False,False,79,91,93,2
3,목포,2016,2016-03-14,-3.0,2016-03-31,-3.0,2016-04-02,-5.0,False,False,False,74,91,93,2
4,백령도,2016,2016-04-11,8.0,2016-04-17,-2.0,2016-04-20,NaN,False,False,True,102,108,111,3


===== 평년비 분석용 최종 데이터 =====
데이터 크기: (159, 15)
연도 범위: 2016 ~ 2025
Location 수: 16
전체 결측치: 0


,0
Location,0
Year,0
Bud_Date,0
Bud_Gap,0
Bloom_Dt,0
Bloom_Gap,0
Full_Dt,0
Full_Gap,0
Bud_Gap_결측여부,0
Bloom_Gap_결측여부,0


,Location,Year,Bud_Date,Bud_Gap,Bloom_Dt,Bloom_Gap,Full_Dt,Full_Gap,Bud_Gap_결측여부,Bloom_Gap_결측여부,Full_Gap_결측여부,Bud_DOY,Bloom_DOY,Full_DOY,Flowering_Duration
0,광주,2016,2016-03-13,0.0,2016-03-29,-2.0,2016-03-31,-4.0,False,False,False,73,89,91,2
1,대구,2016,2016-03-22,6.0,2016-03-25,-4.0,2016-03-31,-2.0,False,False,False,82,85,91,6
2,대전,2016,2016-03-19,-6.0,2016-03-31,-4.0,2016-04-02,-3.0,False,False,False,79,91,93,2
3,목포,2016,2016-03-14,-3.0,2016-03-31,-3.0,2016-04-02,-5.0,False,False,False,74,91,93,2
4,북강릉,2016,2016-03-19,-5.0,2016-04-04,0.0,2016-04-06,-1.0,False,False,False,79,95,97,2


"cherry_blossom_date_clean.csv"= 날짜·DOY·유지기간 분석용 데이터
- Bud_Date, Bloom_Dt, Full_Dt, Bud_DOY, Bloom_DOY, Full_DOY 분석에 사용
- Flowering_Duration은 만발일에서 개화일을 뺀 값으로 계산함
- 평비 결측이 있어도 날짜값이 있으면 행을 유지함
- 연도별 개화 시기 변화, 지역별 개화 시기 비교, 개화 진행 기간 분석에 사용

"cherry_blossom_anomaly_clean.csv"= 평년비·유지기간 분석용 데이터
- Bud_Gap, Bloom_Gap, Full_Gap 분석에 사용
- Flowering_Duration은 만발일에서 개화일을 뺀 값으로 계산함
- 평비 결측 행은 제거함
- 평년보다 빠른지/늦은지 비교하는 분석에 사용
